Modelo LightBGM

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import pickle
import sys
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, make_scorer
import os
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Cargar el archivo
df_final = pd.read_pickle("/Users/aroamateogomez/Desktop/BootcampIA/Proyecto 9/project-ai-data-scientistG2/data/preprocessed_data.pkl")

print("🔍 ANÁLISIS DEL DICCIONARIO:")
print(f"Tipo: {type(df_final)}")
print(f"Claves: {list(df_final.keys())}")

# Analizar cada clave
for key, value in df_final.items():
    print(f"\n📁 {key}:")
    print(f"   Tipo: {type(value)}")
    
    if hasattr(value, 'shape'):
        print(f"   Shape: {value.shape}")
    elif hasattr(value, '__len__'):
        print(f"   Longitud: {len(value)}")
    
    # Mostrar primeros elementos si es array-like
    try:
        if hasattr(value, 'head'):
            print(f"   Primeros valores: {value.head(3).tolist() if hasattr(value.head(3), 'tolist') else value.head(3)}")
        else:
            print(f"   Primeros valores: {value[:3] if len(value) > 3 else value}")
    except:
        print(f"   No se puede mostrar contenido")

In [ ]:
# Definir los datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = df_final['smote'][0], df_final['smote'][1], df_final['smote'][2], df_final['smote'][3]

# Entrenar un modelo (ejemplo con LightGBM)
final_model = lgb.LGBMClassifier(random_state=42)
final_model.fit(X_train, y_train)

# Ahora puedes evaluar
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

# Evaluación del Modelo Optimizado

# Primero verifica qué variables tienes disponibles
print("Variables disponibles:")
print([var for var in dir() if not var.startswith('_')])

# Verifica si tienes modelos entrenados
try:
    # Si tienes un modelo con otro nombre, ajústalo aquí
    # Por ejemplo: model, classifier, best_model, etc.
    modelo_entrenado = final_model  # o cambia por el nombre correcto
    print("✅ Modelo encontrado")
except NameError:
    print("❌ Modelo no encontrado. Necesitas entrenar un modelo primero.")
    
# Si el modelo existe, procede con la evaluación
if 'final_model' in dir() and final_model is not None:
    # 5.1. Hacer predicciones con el modelo optimizado
    y_pred = final_model.predict(X_test)
    y_pred_proba = final_model.predict_proba(X_test)[:, 1]

    # 5.2. Métricas completas
    

    print("📊 EVALUACIÓN DEL MODELO OPTIMIZADO")
    print("=" * 50)

    # Classification Report
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
    # Matriz de Confusión
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Predicho 0', 'Predicho 1'],
                yticklabels=['Real 0', 'Real 1'])
    plt.title('Matriz de Confusión - Modelo Optimizado')
    plt.ylabel('Real')
    plt.xlabel('Predicho')
    plt.show()

    # Métricas adicionales
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"🎯 ROC-AUC Score: {roc_auc:.4f}")

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f"Modelo Optimizado (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Clasificador Aleatorio')
    plt.xlabel('Tasa de Falsos Positivos')
    plt.ylabel('Tasa de Verdaderos Positivos')
    plt.title('Curva ROC - Modelo Optimizado')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Por favor, entrena un modelo primero o verifica el nombre del modelo.")

In [ ]:
# 4.1. Crear y Ejecutar el Estudio
# Mínimo de 100 pruebas es recomendado, pero ajusta según tu tiempo.
N_TRIALS = 150 
STUDY_NAME = "lgbm_recall_optimization"

study = optuna.create_study(direction="minimize", study_name=STUDY_NAME)
print(f"Iniciando estudio de Optuna con {N_TRIALS} pruebas...")

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# 4.2. Mostrar los Resultados
print("\n" + "="*50)
print("🏆 OPTIMIZACIÓN FINALIZADA")
print(f"Mejor valor (Recall Max): {-study.best_value:.4f}")
print("Mejores Hiperparámetros:")
print(study.best_params)
print("="*50)

# 4.3. Entrenar el Modelo Final con los Mejores Parámetros
best_params = study.best_params
best_params['scale_pos_weight'] = SCALE_POS_WEIGHT # Asegurar el peso de clase

# Entrenar en el dataset COMPLETO de entrenamiento (X, y)
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)

# 4.4. Guardar el Modelo Final
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
with open(ARTIFACTS_PATH / MODEL_FILENAME, 'wb') as f:
    pickle.dump(final_model, f)
    
print(f"\n✅ Modelo LightGBM optimizado y guardado en: {ARTIFACTS_PATH / MODEL_FILENAME}")